## GPU-accelerated computations

In this notebook you will see how to:

- accelerate array-level computations using GPU

### Components of earthkit

This tutorial uses the following earthkit components - click any logo to open the package documentation:

<div align="center">
  <a href="https://earthkit-hydro.readthedocs.io/en/latest/" target="_blank" style="display:inline-block; margin: 0 15px;">
    <img src="https://raw.githubusercontent.com/ecmwf/logos/refs/heads/main/logos/earthkit/earthkit-hydro-light.svg" alt="earthkit-hydro" width="200">
  </a>
  <a href="https://earthkit-meteo.readthedocs.io/en/latest/" target="_blank" style="display:inline-block; margin: 0 15px;">
    <img src="https://raw.githubusercontent.com/ecmwf/logos/refs/heads/main/logos/earthkit/earthkit-meteo-light.svg" alt="earthkit-meteo" width="200">
  </a> 
</div>

> Note: The examples in this notebook requires an optional dependency - the Python package **torch**. Prior to running the notebook, install it by uncommenting the line below.

In [43]:
#!pip install torch

In [3]:
import earthkit.hydro as ekh
import torch

# MPS does not support float64, so we set the default dtype to float32
# for fair comparison
torch.default_dtype = torch.float32

### 1. Accelerating earthkit-hydro

By default, computations in earthkit run with numpy, which is CPU-only. To demonstrate a speedup in these examples, we use the tensor library torch instead, which supports multiple devices such as cpu, cuda and mps. We illustrate speedups using mps here.

In [4]:
# load two river networks, one on CPU and one on MPS
torch_network_cpu = ekh.river_network.load("efas", "5").to_device(array_backend="torch", device="cpu")
torch_network_mps = ekh.river_network.load("efas", "5").to_device(array_backend="torch", device="mps")

River network not found in cache (/var/folders/td/yqnxcqpx39dc855vwjtv5hj40000gn/T/tmpgx8ui7eg_earthkit_hydro/1.3_0dc8123bbf944ff1cb86f41bc7506e891baaa990666d836fc0cf2edd503916db.joblib).
River network loaded, saving to cache (/var/folders/td/yqnxcqpx39dc855vwjtv5hj40000gn/T/tmpgx8ui7eg_earthkit_hydro/1.3_0dc8123bbf944ff1cb86f41bc7506e891baaa990666d836fc0cf2edd503916db.joblib).
Loading river network from cache (/var/folders/td/yqnxcqpx39dc855vwjtv5hj40000gn/T/tmpgx8ui7eg_earthkit_hydro/1.3_0dc8123bbf944ff1cb86f41bc7506e891baaa990666d836fc0cf2edd503916db.joblib).


In [5]:
# CPU timings (benchmark)
tensor_cpu = torch.ones(3, *torch_network_cpu.shape, device="cpu")
%timeit ekh.upstream.array.sum(torch_network_cpu, tensor_cpu)

18.2 s ± 210 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
# MPS timings
tensor_mps = torch.ones(3, *torch_network_mps.shape, device="mps")
%timeit ekh.upstream.array.sum(torch_network_mps, tensor_mps)

2.96 s ± 31.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


That's a nice speedup! With larger data, even further speedups can be seen.

Note also that other devices are possible - e.g. with cuda-enabled devices, we unlock further possibilities. See [here](https://earthkit.readthedocs.io/en/latest/tutorials/gpu_hydro.html) for an example showing how to speedup both **both array and xarray** codes by switching from numpy to cupy.

### 2. Accelerating earthkit-meteo

Analagously, we can achieve the same in earthkit-meteo.

In [9]:
# CPU timings (benchmark)
n_samples = 30000
t_cpu = torch.linspace(250, 300, n_samples, device="cpu")
p_cpu = torch.linspace(100000, 50000, n_samples, device="cpu")

%timeit potential_temperature(t_cpu, p_cpu)

335 μs ± 13.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [10]:
# MPS timings
t_mps = torch.linspace(250, 300, n_samples, device="mps")
p_mps = torch.linspace(100000, 50000, n_samples, device="mps")

%timeit potential_temperature(t_mps, p_mps)

22.4 μs ± 90.2 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


Again, with larger data, even further speedups can be seen.